# Pytest test suite for the ai_generated_oscillator.py file from Week 3 Homework

In [1]:
!pytest --version
!python --version

pytest 9.1.1


Python 3.11.16


## 1. Shared fixtures (`conftest.py`)
Selects the module under test through the `OSC_MODULE` environment variable and provides the integration grid and the plotted-lines fixture.

In [2]:
%%writefile conftest.py
"""Shared pytest fixtures for the Week 3 oscillator test suite.

The module under test is chosen with the environment variable OSC_MODULE:

    OSC_MODULE=ai_generated_oscillator_original python -m pytest -v   # buggy
    python -m pytest -v                                               # fixed (default)
"""
import importlib
import os
import sys

import matplotlib

matplotlib.use("Agg")  # draw figures off-screen: no windows pop up during tests

import matplotlib.pyplot as plt
import numpy as np
import pytest

# Make sure the folder that contains this file (and the oscillator scripts) is importable.
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))


@pytest.fixture(scope="session")
def osc():
    """Import the oscillator module named by OSC_MODULE (fixed version by default)."""
    name = os.environ.get("OSC_MODULE", "ai_generated_oscillator_fixed")
    return importlib.import_module(name)


@pytest.fixture(scope="session")
def x_grid():
    """Fine, wide grid for integrals: psi_n decays like exp(-x^2/2), so |x| <= 12 is effectively infinite."""
    return np.linspace(-12.0, 12.0, 4001)


@pytest.fixture
def plotted_lines(osc, monkeypatch):
    """Run osc.main() with plt.show() disabled and return the Line2D objects it drew."""
    plt.close("all")
    monkeypatch.setattr(plt, "show", lambda *args, **kwargs: None)
    osc.main()
    lines = plt.gca().get_lines()
    yield lines
    plt.close("all")


Overwriting conftest.py


## 2. Test suite (`test_oscillator.py`)
Each group of tests maps to one bug in `week03_Aguilar-Mayorga_bugreport.md`, plus an extra orthogonality check.

In [3]:
%%writefile test_oscillator.py
"""Specification-first tests for ai_generated_oscillator.py (Week 3).

Specification (dimensionless units, hbar = m = omega = 1, xi = x):
    E_n   = (n + 1/2) hbar omega                                   Griffiths eq. 2.50
    psi_n = (2^n n!)^(-1/2) pi^(-1/4) H_n(x) exp(-x^2/2)            Griffiths eq. 2.69
    H_n   = physicists' Hermite polynomials, e.g. H_2(x) = 4x^2 - 2
    <psi_m|psi_n> = delta_mn  (normalized and mutually orthogonal)
    main() plots the 5 lowest states n = 0..4 as |psi_n|^2.

Each test maps to one bug in week03_Aguilar-Mayorga_bugreport.md.
"""
import math

import numpy as np
import pytest

N_LEVELS = 5
ATOL_INTEGRAL = 1e-8  # trapezoid rule on x_grid is accurate to ~1e-15; 1e-8 leaves a wide safety margin


# --------------------------------------------------------------------------- #
# Bug #1 — wrong physics formula for the energy levels
# --------------------------------------------------------------------------- #
def test_energy_levels_match_griffiths(osc):
    """E_n = (n + 1/2) for n = 0..4 with hbar = omega = 1 (Griffiths eq. 2.50)."""
    expected = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
    np.testing.assert_allclose(osc.energy_levels(), expected, rtol=1e-12)


def test_zero_point_energy_is_nonzero(osc):
    """The ground state has E_0 = hbar*omega/2 > 0 (zero-point energy), never 0."""
    assert osc.energy_levels()[0] == pytest.approx(0.5)


@pytest.mark.parametrize("hbar, omega", [(1.0, 1.0), (2.0, 3.0), (1.054571817e-34, 1.0e14)])
def test_energy_scales_with_hbar_omega(osc, hbar, omega):
    """E_n must scale as hbar*omega*(n + 1/2) for any physical values of hbar and omega."""
    expected = hbar * omega * (np.arange(N_LEVELS) + 0.5)
    np.testing.assert_allclose(osc.energy_levels(hbar=hbar, omega=omega), expected, rtol=1e-12)


# --------------------------------------------------------------------------- #
# Bug #3 — hallucinated API (special.hermite_poly); must use PHYSICISTS' H_n
# --------------------------------------------------------------------------- #
def test_harmonic_state_runs_and_returns_array(osc):
    """harmonic_state(n, x) must run (no AttributeError) and return one value per grid point."""
    x = np.linspace(-3, 3, 7)
    psi = osc.harmonic_state(2, x)
    assert np.shape(psi) == x.shape
    assert np.all(np.isfinite(psi))


def test_uses_physicists_hermite_n2(osc):
    """psi_2(x) * exp(x^2/2) must be proportional to H_2(x) = 4x^2 - 2 (physicists'),
    not to He_2(x) = x^2 - 1 (probabilists'). Comparing the *shape* (ratio to the value
    at x = 0) makes this test independent of the normalization constant."""
    x = np.array([-2.0, -1.0, -1 / np.sqrt(2), 0.0, 0.5, 1.0, 2.0])
    poly_part = osc.harmonic_state(2, x) * np.exp(x**2 / 2)      # = norm * H_2(x)
    H2 = 4 * x**2 - 2
    np.testing.assert_allclose(poly_part / poly_part[3], H2 / H2[3], rtol=1e-12, atol=1e-12)
    # H_2 vanishes at x = ±1/sqrt(2); He_2 would vanish at x = ±1 instead.
    assert poly_part[2] == pytest.approx(0.0, abs=1e-12)


# --------------------------------------------------------------------------- #
# Bug #4 — incorrect normalization constant
# --------------------------------------------------------------------------- #
@pytest.mark.parametrize("n", range(N_LEVELS))
def test_states_are_normalized(osc, x_grid, n):
    """Integral of |psi_n|^2 over all x equals 1 for n = 0..4 (Griffiths eq. 2.69)."""
    psi = osc.harmonic_state(n, x_grid)
    assert np.trapezoid(np.abs(psi) ** 2, x_grid) == pytest.approx(1.0, abs=ATOL_INTEGRAL)


def test_ground_state_peak_value(osc):
    """psi_0(0) = pi^(-1/4): a pointwise check of the normalization, no integration needed."""
    assert osc.harmonic_state(0, np.array([0.0]))[0] == pytest.approx(np.pi ** -0.25, rel=1e-12)


# --------------------------------------------------------------------------- #
# Extra — orthogonality (of psi_n, i.e. of H_n with weight exp(-x^2))
# --------------------------------------------------------------------------- #
@pytest.mark.parametrize("m, n", [(m, n) for m in range(N_LEVELS) for n in range(m + 1, N_LEVELS)])
def test_states_are_orthogonal(osc, x_grid, m, n):
    """<psi_m|psi_n> = 0 for m != n. Since psi_m psi_n ∝ H_m H_n exp(-x^2), this is the
    orthogonality of the Hermite polynomials with weight exp(-x^2)."""
    overlap = np.trapezoid(osc.harmonic_state(m, x_grid) * osc.harmonic_state(n, x_grid), x_grid)
    assert overlap == pytest.approx(0.0, abs=ATOL_INTEGRAL)


def test_hermite_weighted_orthogonality_constants(osc, x_grid):
    """Full Gram matrix of H_n = psi_n * exp(x^2/2) / norm_n with weight exp(-x^2) must be
    diag(sqrt(pi) 2^n n!). Built from the module's psi_n, so it checks H_n and norm together."""
    norms = np.array([1 / math.sqrt(2**n * math.factorial(n) * math.sqrt(math.pi)) for n in range(N_LEVELS)])
    psis = np.array([osc.harmonic_state(n, x_grid) for n in range(N_LEVELS)])
    gram = np.trapezoid(psis[:, None, :] * psis[None, :, :], x_grid, axis=-1)   # <psi_m|psi_n>
    hermite_gram = gram / np.outer(norms, norms)                                 # ∫ H_m H_n e^{-x^2} dx
    expected = np.diag([math.sqrt(math.pi) * 2**n * math.factorial(n) for n in range(N_LEVELS)])
    np.testing.assert_allclose(hermite_gram, expected, atol=1e-6)


@pytest.mark.parametrize("n", range(N_LEVELS))
def test_parity(osc, n):
    """psi_n(-x) = (-1)^n psi_n(x): even states are symmetric, odd states antisymmetric."""
    x = np.linspace(0.1, 4.0, 40)
    np.testing.assert_allclose(osc.harmonic_state(n, -x), (-1) ** n * osc.harmonic_state(n, x), atol=1e-12)


# --------------------------------------------------------------------------- #
# Bug #2 — off-by-one: main() must plot all 5 states, starting at n = 0
# --------------------------------------------------------------------------- #
def test_main_plots_five_levels(plotted_lines):
    """main() draws exactly one curve per level n = 0..4."""
    assert len(plotted_lines) == N_LEVELS


def test_main_first_curve_is_ground_state(osc, plotted_lines):
    """The first curve is |psi_0|^2 (not psi_1): catches states[1:] even if labels say n=0."""
    line = plotted_lines[0]
    x = line.get_xdata()
    np.testing.assert_allclose(line.get_ydata(), np.abs(osc.harmonic_state(0, x)) ** 2, atol=1e-12)
    assert line.get_label() == "n=0"


# --------------------------------------------------------------------------- #
# Bug #5 — wrong output quantity: must plot |psi|^2, not psi
# --------------------------------------------------------------------------- #
def test_plotted_curves_are_probability_densities(osc, plotted_lines):
    """Every curve must equal |psi_n|^2: non-negative everywhere and matching the label's n."""
    for n, line in enumerate(plotted_lines):
        x, y = line.get_xdata(), line.get_ydata()
        assert np.all(y >= 0), f"curve n={n} has negative values: it is psi, not |psi|^2"
        np.testing.assert_allclose(y, np.abs(osc.harmonic_state(n, x)) ** 2, atol=1e-12)


Overwriting test_oscillator.py


## 3. Run against the fixed version (`ai_generated_oscillator_fixed.py`)
Expected: all tests pass.

In [4]:
import sys
!{sys.executable} -m pytest -v test_oscillator.py

============================= test session starts ==============================
platform darwin -- Python 3.11.16, pytest-9.1.1, pluggy-1.6.0 -- /opt/anaconda3/envs/ai-physics/bin/python
cachedir: .pytest_cache
rootdir: /Users/mayorga/Documents/PhD_UoM/Classes/AI/ai-physics-Aguilar-Mayorga/Hwks/week3
plugins: typeguard-4.6.0, anyio-4.15.1
collected 32 items                                                             

test_oscillator.py::test_energy_levels_match_griffiths 

PASSED            [  3%]
test_oscillator.py::test_zero_point_energy_is_nonzero PASSED             [  6%]
test_oscillator.py::test_energy_scales_with_hbar_omega[1.0-1.0] PASSED   [  9%]
test_oscillator.py::test_energy_scales_with_hbar_omega[2.0-3.0] PASSED   [ 12%]
test_oscillator.py::test_energy_scales_with_hbar_omega[1.054571817e-34-100000000000000.0] PASSED [ 15%]
test_oscillator.py::test_harmonic_state_runs_and_returns_array 

PASSED    [ 18%]
test_oscillator.py::test_uses_physicists_hermite_n2 PASSED               [ 21%]
test_oscillator.py::test_states_are_normalized[0] PASSED                 [ 25%]
test_oscillator.py::test_states_are_normalized[1] PASSED                 [ 28%]
test_oscillator.py::test_states_are_normalized[2] PASSED                 [ 31%]
test_oscillator.py::test_states_are_normalized[3] PASSED                 [ 34%]
test_oscillator.py::test_states_are_normalized[4] PASSED                 [ 37%]
test_oscillator.py::test_ground_state_peak_value PASSED                  [ 40%]
test_oscillator.py::test_states_are_orthogonal[0-1] PASSED               [ 43%]
test_oscillator.py::test_states_are_orthogonal[0-2] PASSED               [ 46%]
test_oscillator.py::test_states_are_orthogonal[0-3] PASSED               [ 50%]
test_oscillator.py::test_states_are_orthogonal[0-4] PASSED               [ 53%]
test_oscillator.py::test_states_are_orthogonal[1-2] PASSED               [ 56%]
test_oscillator.py::tes

PASSED                   [ 93%]
test_oscillator.py::test_main_first_curve_is_ground_state PASSED         [ 96%]
test_oscillator.py::test_plotted_curves_are_probability_densities PASSED [100%]

============================== 32 passed in 0.24s ==============================


## 4. Run against the original buggy version (`ai_generated_oscillator_original.py`)
Expected: every test fails (or errors); the `AttributeError` from `special.hermite_poly` masks the other wavefunction bugs, while the energy tests fail on their own.

In [5]:
!OSC_MODULE=ai_generated_oscillator_original {sys.executable} -m pytest -v test_oscillator.py

============================= test session starts ==============================
platform darwin -- Python 3.11.16, pytest-9.1.1, pluggy-1.6.0 -- /opt/anaconda3/envs/ai-physics/bin/python
cachedir: .pytest_cache
rootdir: /Users/mayorga/Documents/PhD_UoM/Classes/AI/ai-physics-Aguilar-Mayorga/Hwks/week3
plugins: typeguard-4.6.0, anyio-4.15.1
collected 32 items                                                             

test_oscillator.py::test_energy_levels_match_griffiths 

FAILED            [  3%]
test_oscillator.py::test_zero_point_energy_is_nonzero FAILED             [  6%]
test_oscillator.py::test_energy_scales_with_hbar_omega[1.0-1.0] FAILED   [  9%]
test_oscillator.py::test_energy_scales_with_hbar_omega[2.0-3.0] FAILED   [ 12%]
test_oscillator.py::test_energy_scales_with_hbar_omega[1.054571817e-34-100000000000000.0] FAILED [ 15%]
test_oscillator.py::test_harmonic_state_runs_and_returns_array FAILED    [ 18%]
test_oscillator.py::test_uses_physicists_hermite_n2 FAILED               [ 21%]
test_oscillator.py::test_states_are_normalized[0] FAILED                 [ 25%]
test_oscillator.py::test_states_are_normalized[1] FAILED                 [ 28%]
test_oscillator.py::test_states_are_normalized[2] FAILED                 [ 31%]
test_oscillator.py::test_states_are_normalized[3] FAILED                 [ 34%]
test_oscillator.py::test_states_are_normalized[4] FAILED                 [ 37%]
test_oscillator.py::test_ground_state_peak_value FAILED                

FAILED               [ 59%]
test_oscillator.py::test_states_are_orthogonal[1-4] FAILED               [ 62%]
test_oscillator.py::test_states_are_orthogonal[2-3] FAILED               [ 65%]
test_oscillator.py::test_states_are_orthogonal[2-4] FAILED               [ 68%]
test_oscillator.py::test_states_are_orthogonal[3-4] FAILED               [ 71%]
test_oscillator.py::test_hermite_weighted_orthogonality_constants FAILED [ 75%]
test_oscillator.py::test_parity[0] FAILED                                [ 78%]
test_oscillator.py::test_parity[1] FAILED                                [ 81%]
test_oscillator.py::test_parity[2] FAILED                                [ 84%]
test_oscillator.py::test_parity[3] FAILED                                [ 87%]
test_oscillator.py::test_parity[4] FAILED                                [ 90%]
test_oscillator.py::test_main_plots_five_levels ERROR                    [ 93%]
test_oscillator.py::test_main_first_curve_is_ground_state ERROR          [ 96%]
test_oscilla

=========================== short test summary info ============================
FAILED test_oscillator.py::test_energy_levels_match_griffiths - AssertionError: 
FAILED test_oscillator.py::test_zero_point_energy_is_nonzero - assert np.float64(0.0) == 0.5 ± 5.0e-07
FAILED test_oscillator.py::test_energy_scales_with_hbar_omega[1.0-1.0] - AssertionError: 
FAILED test_oscillator.py::test_energy_scales_with_hbar_omega[2.0-3.0] - AssertionError: 
FAILED test_oscillator.py::test_energy_scales_with_hbar_omega[1.054571817e-34-100000000000000.0] - AssertionError: 
FAILED test_oscillator.py::test_harmonic_state_runs_and_returns_array - AttributeError: module 'scipy.special' has no attribute 'hermite_poly'
FAILED test_oscillator.py::test_uses_physicists_hermite_n2 - AttributeError: module 'scipy.special' has no attribute 'hermite_poly'
FAILED test_oscillator.py::test_states_are_normalized[0] - AttributeError: module 'scipy.special' has no attribute 'hermite_poly'
FAILED test_oscillator.py::test_st

# --------------------------------------------------------------------------- #
# Test-driven prompt strategy
# --------------------------------------------------------------------------- #

To try to mimic the test-driven coding approach to coding with an llm described by Mathews and Nagappan, I used the following prompt using the test suite previously created. 
**Note: I left out the "Extra" tests from the prompt so I could have a test that the llm never saw to test it against, as described in *https://arxiv.org/pdf/2402.13521***

### Prompt used:
"Please write a Python script, from scratch, that computes the first five energy
levels of the 1D quantum harmonic oscillator and plots the probability densities
|psi_n(x)|^2 for n = 0..4 using numpy, scipy and matplotlib.

Requirements:
- Python 3.11, numpy 2.x. Dimensionless units (hbar = m = omega = 1 by default).
- The module will be imported as `AI_test_driven_oscillator`.
- It must pass every test in the pytest suite below without modifying the tests.
  The tests are the physical specification: do not special-case or hard-code
  values to make them pass.
- Reply with one complete .py file only.

conftest.py:
```python
"""Shared pytest fixtures for the Week 3 oscillator test suite.

The module under test is chosen with the environment variable OSC_MODULE:

    OSC_MODULE=ai_generated_oscillator_original python -m pytest -v   # buggy
    python -m pytest -v                                               # fixed (default)
"""
import importlib
import os
import sys

import matplotlib

matplotlib.use("Agg")  # draw figures off-screen: no windows pop up during tests

import matplotlib.pyplot as plt
import numpy as np
import pytest

# Make sure the folder that contains this file (and the oscillator scripts) is importable.
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))


@pytest.fixture(scope="session")
def osc():
    """Import the oscillator module named by OSC_MODULE (fixed version by default)."""
    name = os.environ.get("OSC_MODULE", "ai_generated_oscillator_fixed")
    return importlib.import_module(name)


@pytest.fixture(scope="session")
def x_grid():
    """Fine, wide grid for integrals: psi_n decays like exp(-x^2/2), so |x| <= 12 is effectively infinite."""
    return np.linspace(-12.0, 12.0, 4001)


@pytest.fixture
def plotted_lines(osc, monkeypatch):
    """Run osc.main() with plt.show() disabled and return the Line2D objects it drew."""
    plt.close("all")
    monkeypatch.setattr(plt, "show", lambda *args, **kwargs: None)
    osc.main()
    lines = plt.gca().get_lines()
    yield lines
    plt.close("all")
```

test_oscillator.py:
```python
"""Specification-first tests for ai_generated_oscillator.py (Week 3).

Specification (dimensionless units, hbar = m = omega = 1, xi = x):
    E_n   = (n + 1/2) hbar omega                                   Griffiths eq. 2.50
    psi_n = (2^n n!)^(-1/2) pi^(-1/4) H_n(x) exp(-x^2/2)            Griffiths eq. 2.69
    H_n   = physicists' Hermite polynomials, e.g. H_2(x) = 4x^2 - 2
    <psi_m|psi_n> = delta_mn  (normalized and mutually orthogonal)
    main() plots the 5 lowest states n = 0..4 as |psi_n|^2.

Each test maps to one bug in week03_Aguilar-Mayorga_bugreport.md.
"""
import math

import numpy as np
import pytest

N_LEVELS = 5
ATOL_INTEGRAL = 1e-8  # trapezoid rule on x_grid is accurate to ~1e-15; 1e-8 leaves a wide safety margin


# --------------------------------------------------------------------------- #
# Bug #1 — wrong physics formula for the energy levels
# --------------------------------------------------------------------------- #
def test_energy_levels_match_griffiths(osc):
    """E_n = (n + 1/2) for n = 0..4 with hbar = omega = 1 (Griffiths eq. 2.50)."""
    expected = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
    np.testing.assert_allclose(osc.energy_levels(), expected, rtol=1e-12)


def test_zero_point_energy_is_nonzero(osc):
    """The ground state has E_0 = hbar*omega/2 > 0 (zero-point energy), never 0."""
    assert osc.energy_levels()[0] == pytest.approx(0.5)


@pytest.mark.parametrize("hbar, omega", [(1.0, 1.0), (2.0, 3.0), (1.054571817e-34, 1.0e14)])
def test_energy_scales_with_hbar_omega(osc, hbar, omega):
    """E_n must scale as hbar*omega*(n + 1/2) for any physical values of hbar and omega."""
    expected = hbar * omega * (np.arange(N_LEVELS) + 0.5)
    np.testing.assert_allclose(osc.energy_levels(hbar=hbar, omega=omega), expected, rtol=1e-12)


# --------------------------------------------------------------------------- #
# Bug #3 — hallucinated API (special.hermite_poly); must use PHYSICISTS' H_n
# --------------------------------------------------------------------------- #
def test_harmonic_state_runs_and_returns_array(osc):
    """harmonic_state(n, x) must run (no AttributeError) and return one value per grid point."""
    x = np.linspace(-3, 3, 7)
    psi = osc.harmonic_state(2, x)
    assert np.shape(psi) == x.shape
    assert np.all(np.isfinite(psi))


def test_uses_physicists_hermite_n2(osc):
    """psi_2(x) * exp(x^2/2) must be proportional to H_2(x) = 4x^2 - 2 (physicists'),
    not to He_2(x) = x^2 - 1 (probabilists'). Comparing the *shape* (ratio to the value
    at x = 0) makes this test independent of the normalization constant."""
    x = np.array([-2.0, -1.0, -1 / np.sqrt(2), 0.0, 0.5, 1.0, 2.0])
    poly_part = osc.harmonic_state(2, x) * np.exp(x**2 / 2)      # = norm * H_2(x)
    H2 = 4 * x**2 - 2
    np.testing.assert_allclose(poly_part / poly_part[3], H2 / H2[3], rtol=1e-12, atol=1e-12)
    # H_2 vanishes at x = ±1/sqrt(2); He_2 would vanish at x = ±1 instead.
    assert poly_part[2] == pytest.approx(0.0, abs=1e-12)


# --------------------------------------------------------------------------- #
# Bug #4 — incorrect normalization constant
# --------------------------------------------------------------------------- #
@pytest.mark.parametrize("n", range(N_LEVELS))
def test_states_are_normalized(osc, x_grid, n):
    """Integral of |psi_n|^2 over all x equals 1 for n = 0..4 (Griffiths eq. 2.69)."""
    psi = osc.harmonic_state(n, x_grid)
    assert np.trapezoid(np.abs(psi) ** 2, x_grid) == pytest.approx(1.0, abs=ATOL_INTEGRAL)


def test_ground_state_peak_value(osc):
    """psi_0(0) = pi^(-1/4): a pointwise check of the normalization, no integration needed."""
    assert osc.harmonic_state(0, np.array([0.0]))[0] == pytest.approx(np.pi ** -0.25, rel=1e-12)


# --------------------------------------------------------------------------- #
# Bug #2 — off-by-one: main() must plot all 5 states, starting at n = 0
# --------------------------------------------------------------------------- #
def test_main_plots_five_levels(plotted_lines):
    """main() draws exactly one curve per level n = 0..4."""
    assert len(plotted_lines) == N_LEVELS


def test_main_first_curve_is_ground_state(osc, plotted_lines):
    """The first curve is |psi_0|^2 (not psi_1): catches states[1:] even if labels say n=0."""
    line = plotted_lines[0]
    x = line.get_xdata()
    np.testing.assert_allclose(line.get_ydata(), np.abs(osc.harmonic_state(0, x)) ** 2, atol=1e-12)
    assert line.get_label() == "n=0"


# --------------------------------------------------------------------------- #
# Bug #5 — wrong output quantity: must plot |psi|^2, not psi
# --------------------------------------------------------------------------- #
def test_plotted_curves_are_probability_densities(osc, plotted_lines):
    """Every curve must equal |psi_n|^2: non-negative everywhere and matching the label's n."""
    for n, line in enumerate(plotted_lines):
        x, y = line.get_xdata(), line.get_ydata()
        assert np.all(y >= 0), f"curve n={n} has negative values: it is psi, not |psi|^2"
        np.testing.assert_allclose(y, np.abs(osc.harmonic_state(n, x)) ** 2, atol=1e-12)
```
"


# Test-driven script for the Quantum Harmonic Oscillator obtained using the previously shown prompt, with ChatGPT, using GPT-6:

In [6]:
%%writefile AI_test_driven_oscillator.py
"""
1D Quantum Harmonic Oscillator
------------------------------
Computes the first five energy levels and plots the corresponding
probability densities |psi_n(x)|^2 for n = 0, 1, 2, 3, 4.

Dimensionless units are used by default:
    hbar = m = omega = 1

The module can also be imported as AI_test_driven_oscillator.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import eval_hermite, factorial


# ---------------------------------------------------------------------
# Physical constants and configuration
# ---------------------------------------------------------------------

HBAR = 1.0
MASS = 1.0
OMEGA = 1.0

N_LEVELS = 5


# ---------------------------------------------------------------------
# Energy eigenvalues
# ---------------------------------------------------------------------

def energy_levels(hbar=HBAR, omega=OMEGA):
    """
    Calculate the first five energy eigenvalues of the 1D
    quantum harmonic oscillator.

    Parameters
    ----------
    hbar : float
        Reduced Planck constant.
    omega : float
        Angular frequency of the oscillator.

    Returns
    -------
    numpy.ndarray
        Energy eigenvalues E_n = hbar * omega * (n + 1/2).
    """
    n = np.arange(N_LEVELS)
    return hbar * omega * (n + 0.5)


# ---------------------------------------------------------------------
# Harmonic oscillator eigenstates
# ---------------------------------------------------------------------

def harmonic_state(n, x, hbar=HBAR, mass=MASS, omega=OMEGA):
    """
    Evaluate the normalized nth eigenstate of the 1D quantum
    harmonic oscillator.

    The normalized wavefunction is

        psi_n(x) = (1 / sqrt(2^n n!))
                   * (m*omega / (pi*hbar))^(1/4)
                   * H_n(xi)
                   * exp(-xi^2 / 2)

    where

        xi = sqrt(m*omega / hbar) * x

    and H_n is the physicists' Hermite polynomial.

    Parameters
    ----------
    n : int
        Quantum number (n >= 0).
    x : array_like
        Spatial coordinates.
    hbar : float
        Reduced Planck constant.
    mass : float
        Particle mass.
    omega : float
        Angular frequency.

    Returns
    -------
    numpy.ndarray
        Normalized wavefunction evaluated at x.
    """
    if not isinstance(n, (int, np.integer)) or n < 0:
        raise ValueError("The quantum number n must be a non-negative integer.")

    if hbar <= 0 or mass <= 0 or omega <= 0:
        raise ValueError("hbar, mass, and omega must be positive.")

    x = np.asarray(x, dtype=float)

    # Dimensionless position coordinate
    xi = np.sqrt(mass * omega / hbar) * x

    # Physicists' Hermite polynomial H_n(xi)
    hermite = eval_hermite(n, xi)

    # Normalization constant
    normalization = (
        (mass * omega / (np.pi * hbar)) ** 0.25
        / np.sqrt((2.0 ** n) * factorial(n))
    )

    # Normalized eigenfunction
    psi = normalization * hermite * np.exp(-xi**2 / 2.0)

    return psi


# ---------------------------------------------------------------------
# Probability densities and plotting
# ---------------------------------------------------------------------

def main():
    """
    Compute and plot the probability densities of the first five
    harmonic oscillator eigenstates.
    """

    # Spatial grid
    x = np.linspace(-5.0, 5.0, 1000)

    # Energy eigenvalues
    energies = energy_levels()

    print("1D Quantum Harmonic Oscillator")
    print("--------------------------------")
    print("Energy eigenvalues (dimensionless units):")

    for n, energy in enumerate(energies):
        print(f"E_{n} = {energy:.6f}")

    # Plot probability densities
    plt.figure(figsize=(9, 6))

    for n in range(N_LEVELS):
        psi = harmonic_state(n, x)

        # Probability density
        probability_density = np.abs(psi) ** 2

        plt.plot(
            x,
            probability_density,
            label=f"n={n}"
        )

    plt.xlabel("Position x")
    plt.ylabel(r"Probability density $|\psi_n(x)|^2$")
    plt.title("1D Quantum Harmonic Oscillator")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.show()


# ---------------------------------------------------------------------
# Script entry point
# ---------------------------------------------------------------------

if __name__ == "__main__":
    main()


Overwriting AI_test_driven_oscillator.py


# Using the **complete** test suite (i.e. including the Extra tests) to evaluate the obtained script:

In [7]:
!OSC_MODULE=AI_test_driven_oscillator {sys.executable} -m pytest -v test_oscillator.py

============================= test session starts ==============================
platform darwin -- Python 3.11.16, pytest-9.1.1, pluggy-1.6.0 -- /opt/anaconda3/envs/ai-physics/bin/python
cachedir: .pytest_cache
rootdir: /Users/mayorga/Documents/PhD_UoM/Classes/AI/ai-physics-Aguilar-Mayorga/Hwks/week3
plugins: typeguard-4.6.0, anyio-4.15.1
collected 32 items                                                             

test_oscillator.py::test_energy_levels_match_griffiths 

PASSED            [  3%]
test_oscillator.py::test_zero_point_energy_is_nonzero PASSED             [  6%]
test_oscillator.py::test_energy_scales_with_hbar_omega[1.0-1.0] PASSED   [  9%]
test_oscillator.py::test_energy_scales_with_hbar_omega[2.0-3.0] PASSED   [ 12%]
test_oscillator.py::test_energy_scales_with_hbar_omega[1.054571817e-34-100000000000000.0] PASSED [ 15%]
test_oscillator.py::test_harmonic_state_runs_and_returns_array PASSED    [ 18%]
test_oscillator.py::test_uses_physicists_hermite_n2 PASSED               [ 21%]
test_oscillator.py::test_states_are_normalized[0] PASSED                 [ 25%]
test_oscillator.py::test_states_are_normalized[1] PASSED                 [ 28%]
test_oscillator.py::test_states_are_normalized[2] PASSED                 [ 31%]
test_oscillator.py::test_states_are_normalized[3] PASSED                 [ 34%]
test_oscillator.py::test_states_are_normalized[4] PASSED                 [ 37%]
test_oscillator.py::test_ground_state_peak_value PASSED                

PASSED                   [ 93%]


test_oscillator.py::test_main_first_curve_is_ground_state PASSED         [ 96%]
test_oscillator.py::test_plotted_curves_are_probability_densities PASSED [100%]

============================== 32 passed in 0.20s ==============================


## Iterative refinement
All 32 tests passed on the first attempt, including the three held-out "Extra" tests the model never saw, so no round of iterative refinement was needed and there was no correction to classify as fixing the physical root cause or only the Python error.

# Final reflection:

As seen clearly in this exercise, using a test suite as the specification when asking an llm to create code to solve different tasks/problems, involves undoubtedly the skill of the prompter to understand the problem, its implications, assumptions required to solve it, conventions and much more. But it is just as important to implement this knowledge properly and precisely when writing the test suite, since the resulting code will really only be as good as the specification and tests used for it. For example, in my case I wrongly stated I wanted the code to work out "the first 5 energy levels", instead of generalizing to ask for a more robust code that could calculate n energy values, not only 5. Nevertheless, the model I used did create a code that allows the user to change this value, although it left out this parameter as an argument for the function that calculates them (it is specified as a global variable instead).

Overall, this assignment helped me understand the test-driven strategy and be more conscious of how many details I tend to omit when prompting. 